# Authority IDs and CMIF for letter metadata with an LLM on the DHinfra cluster

Notebook 1 gives names. A CMIF file for [correspSearch](https://correspsearch.net) needs
identifiers: a GND number for the writer and a GeoNames ID for the place. In the full pipeline
the names are looked up in the GND (through [lobid-gnd](https://lobid.org/gnd)) and in
[GeoNames](https://www.geonames.org). Where a lookup returns exactly one record, that record is
taken. Where it returns several (fourteen men named Frank, three places called Trieste), the
model reads the letter and picks one, or answers `unclear`.

This notebook does the picking, compares the final IDs with the reference, and writes one CMIF
`<correspDesc>` per letter.

**The lookups are not run here.** GeoNames needs a personal account, and lookup results change
over time. `data/lookups.json` holds the results of the original run for the ten letters of
notebook 1: the extracted values (by `qwen3.5-397b` in thinking mode) and what GND and GeoNames
returned for them. So every run of this notebook starts from the same point.

**Steps:** 1. set up, 2. the data and the prompt, 3. one choice, 4. all choices,
5. compare the IDs with the reference, 6. write CMIF.

## 1. Set up

In [ ]:
# Run once per session. The cluster's Python environment may already have these.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

Your project token is read from `DHINFRA_KEY`, either from the environment or from a `.env`
file next to this notebook (`DHINFRA_KEY="..."`). Never commit the `.env` file.

In [ ]:
import json
from pathlib import Path
import concurrent.futures as cf

import pandas as pd

import helpers

MODEL = "deepseek-v4.1-flash"   # pick a name from the list printed below
THINKING = False                # True: the model reasons first (slower, more tokens)
WORKERS = 4                     # parallel requests; keep this small, the cluster is shared

PROMPT_DIR = Path("prompts/disambiguation")
LETTER_DIR = Path("data/letters")
LOOKUP_FILE = Path("data/lookups.json")
REFERENCE_FILE = Path("data/reference.json")
OUTPUT_DIR = Path("output") / MODEL
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = helpers.get_client()
print([m.id for m in client.models.list().data])

## 2. The data and the prompt

For each letter, `data/lookups.json` has the saved answer of the first step (`stage1`, the same
JSON format as in notebook 1) and the lookup result for the writer and for the place:

- `unique`: one record found, its ID is taken as it is.
- `ambiguous`: several records found, they are listed under `candidates`. The model has to choose.
- `not_found`: no record found.
- `skipped`: the first step answered `unk`, so there was nothing to look up.

In [ ]:
lookups = json.loads(LOOKUP_FILE.read_text(encoding="utf-8"))
overview = []
for entry in lookups:
    overview.append({
        "letter": entry["letter"],
        "writer": helpers.writer_name(entry["stage1"]),
        "writer_lookup": entry["writer"]["status"],
        "gnd_candidates": len(entry["writer"].get("candidates", [])),
        "place": helpers.place_name(entry["stage1"]),
        "place_lookup": entry["place"]["status"],
        "geonames_candidates": len(entry["place"].get("candidates", [])),
    })
pd.DataFrame(overview).set_index("letter")

The system prompt has the same three parts as in notebook 1: `prompt.txt` (the task),
`encoding_rules.txt` (how to choose, when to answer `unclear`, the certainty levels) and
`few_shot_examples.txt`. The user message is one JSON object: the name as extracted, the
candidates, the date and the full letter text.

In [ ]:
system_prompt = helpers.build_system_prompt(PROMPT_DIR)
print(f"{len(system_prompt):,} characters")
print(system_prompt[:2000], "...")

## 3. One choice

In `letter1385` the first step found the surname "Frank" and no forename. The GND lookup
returned fourteen candidates named Frank, all alive in 1812: physicians, a painter, a
politician, an orientalist and others. The letter has to decide.

In [ ]:
by_letter = {entry["letter"]: entry for entry in lookups}

def letter_text(name):
    return (LETTER_DIR / f"{name}.txt").read_text(encoding="utf-8")

message = helpers.build_choice_message(by_letter["letter1385"], "writer", letter_text("letter1385"))
print(message[:2500], "...")

In [ ]:
raw, usage = helpers.ask_model(client, MODEL, system_prompt, message, thinking=THINKING)
print(raw)
print()
print("tokens in/out:", usage.prompt_tokens, usage.completion_tokens)

## 4. All choices

One request per ambiguous field, a few at a time. The answers are saved to
`output/<model>/choices.json`.

In [ ]:
tasks = [(entry["letter"], field) for entry in lookups for field in ("writer", "place")
         if entry[field]["status"] == "ambiguous"]
print(len(tasks), "choices to make:", tasks)

def run_one(task):
    name, field = task
    message = helpers.build_choice_message(by_letter[name], field, letter_text(name))
    raw, usage = helpers.ask_model(client, MODEL, system_prompt, message, thinking=THINKING)
    return raw, usage.prompt_tokens, usage.completion_tokens

choices = {}
with cf.ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(run_one, t): t for t in tasks}
    for future in cf.as_completed(futures):
        task = futures[future]
        try:
            raw = future.result()[0]
            choices[task] = helpers.parse_json_answer(raw)
            print("done  ", *task)
        except Exception as err:   # keep going; one failed request should not stop the batch
            print("FAILED", *task, "-", err)

(OUTPUT_DIR / "choices.json").write_text(json.dumps(
    {f"{name}/{field}": answer for (name, field), answer in sorted(choices.items())},
    ensure_ascii=False, indent=2), encoding="utf-8")

## 5. Compare the IDs with the reference

For each letter and field, the final ID is the lookup result where it was `unique`, the model's
choice where it was `ambiguous`, and none otherwise. It is compared with the GND and GeoNames
IDs in `data/reference.json`. The outcomes are those of notebook 1, plus two for fields where
the reference has no ID: some writers have no GND record, and the edition leaves some places
open.

| outcome | meaning |
|---|---|
| correct | same ID as the reference |
| correct (no ID) | no ID, and the reference has none either |
| correct (not in letter) | no ID, and the letter does not give the name |
| missed | no ID, but the reference has one and the letter gives the name |
| wrong | a different ID |
| check (no ID in reference) | an ID where the reference has none. Read it. |

In [ ]:
reference = {r["letter"]: r for r in json.loads(REFERENCE_FILE.read_text(encoding="utf-8"))}

final = {}
rows = []
for entry in lookups:
    name = entry["letter"]
    row = {"letter": name}
    for field, id_name in (("writer", "gnd"), ("place", "geonames")):
        lookup = entry[field]
        label = helpers.writer_name(entry["stage1"]) if field == "writer" else helpers.place_name(entry["stage1"])
        found_id, cert, source = lookup.get(id_name), None, lookup["status"]
        if lookup["status"] == "ambiguous":
            found_id, chosen_label, cert = helpers.read_choice(choices.get((name, field)), field)
            label = chosen_label or label
            source = "model"
        final[(name, field)] = (label, found_id)
        row.update({field: label, f"{id_name}": found_id, f"{field}_from": source,
                    f"{field}_cert": cert,
                    f"{field}_result": helpers.score_id(found_id, reference[name][field])})
    rows.append(row)

table = pd.DataFrame(rows).set_index("letter")
table

In [ ]:
print("writer:\n" + table.writer_result.value_counts().to_string())
print()
print("place:\n" + table.place_result.value_counts().to_string())
print()
print("only the fields the model decided:")
for field in ("writer", "place"):
    decided = table[table[f"{field}_from"] == "model"]
    print(f"  {field}: " + ", ".join(f"{n} {r}" for n, r in decided[f"{field}_result"].items()))

## 6. Write CMIF

Each letter gets a `<correspDesc>` with the sending side: the writer with a link to the GND, the
place with a link to GeoNames, and the date. Values that are unknown are left out. The file is
`output/<model>/correspDesc.xml`.

The addressee is not part of this task (it is usually Hammer-Purgstall himself). A complete
CMIF entry needs a `<correspAction type="received">` as well.

In [ ]:
descs = []
for entry in lookups:
    name = entry["letter"]
    writer, gnd = final[(name, "writer")]
    place, geonames = final[(name, "place")]
    date = helpers.field_value(entry["stage1"].get("date"))
    descs.append(helpers.corresp_desc(name, writer, gnd, place, geonames, date))

xml = helpers.profile_desc(descs)
(OUTPUT_DIR / "correspDesc.xml").write_text(xml, encoding="utf-8")
print(xml)

## Where next

- Change `MODEL` or `THINKING`, or the rules in `prompts/disambiguation/`, and run again.
- To link your own letters, run the lookups first. lobid-gnd can be queried without an account
  (`https://lobid.org/gnd/search?q=Frank&filter=type:Person&format=json`). The GeoNames search
  API needs a free user name. Save the candidates in the format of `data/lookups.json`.
- The date and the language come straight from the first step: they need no lookup.